# Mage-Flow-Turbo — PyTorch BF16 on Kaggle Tesla T4 ×2
## Public bilingual production demo / Demo production công khai song ngữ

### English
This notebook demonstrates one **single text-to-image trajectory** using one logical Mage-Flow-Turbo model distributed explicitly across **two Tesla T4 GPUs**. The runtime uses the qualified upstream PyTorch path with **BF16 dtype/materialization**, SDPA attention, one model load, explicit transformer placement, and fail-closed acceptance gates.

This public notebook is a reproducible demonstration and is not a replacement authority. The separate frozen R2G authority notebook/evidence remains the formal audit artifact.

### Tiếng Việt
Notebook này trình diễn **một luồng text-to-image duy nhất** bằng một model Mage-Flow-Turbo logic được đặt tường minh trên **hai GPU Tesla T4**. Runtime sử dụng pipeline PyTorch upstream đã được kiểm định với **BF16 dtype/materialization**, SDPA attention, chỉ load model một lần, placement transformer tường minh và các acceptance gate fail-closed.

Notebook public này dùng để demo và tái lập kết quả. Notebook/evidence R2G đã đóng băng vẫn là artifact kiểm định chính thức.

| Setting / Thiết lập | Value / Giá trị |
|---|---|
| Runtime | Upstream PyTorch |
| Precision / Độ chính xác | BF16 dtype/materialization |
| GPU | Tesla T4 ×2 |
| Topology | One logical model / one T2I trajectory |
| Transformer split | block 0 → GPU0; blocks 1–11 → GPU1 |
| Attention | SDPA |
| Output | 512×512 RGB |


## Before you run / Trước khi chạy

### English
Prepare a **fresh Kaggle Notebook** with:

1. **Accelerator:** Tesla T4 ×2.
2. **Internet:** ON. The notebook clones the public source repository and fetches the exact pinned upstream Mage source plus one pinned bootstrap wheel.
3. **Kaggle Model:** attach `dangkhoa2016/mage-flow-community-mage-flow-turbo`.
   Kaggle should mount it read-only at:
   `/kaggle/input/models/dangkhoa2016/mage-flow-community-mage-flow-turbo/pytorch/default/1`
4. **Dataset:** no additional dataset is required for this text-to-image demo.
5. Start from a fresh kernel and use **Run All exactly once**.

You do **not** need to upload any project ZIP, identity JSON, or source archive. The project source is cloned automatically from GitHub into `/kaggle/working`.

### Tiếng Việt
Hãy chuẩn bị một **Kaggle Notebook mới** với:

1. **Accelerator:** Tesla T4 ×2.
2. **Internet:** ON. Notebook sẽ tự clone source public từ GitHub và lấy đúng upstream Mage source cùng một bootstrap wheel đã pin.
3. **Kaggle Model:** attach `dangkhoa2016/mage-flow-community-mage-flow-turbo`.
   Kaggle dự kiến mount read-only tại:
   `/kaggle/input/models/dangkhoa2016/mage-flow-community-mage-flow-turbo/pytorch/default/1`
4. **Dataset:** demo text-to-image này không cần dataset bổ sung.
5. Dùng fresh kernel và **Run All đúng một lần**.

Bạn **không cần** upload project ZIP, identity JSON hay source archive nào. Source của dự án được notebook tự động clone từ GitHub vào `/kaggle/working`.


## Architecture / Kiến trúc

### English
The text encoder runs on GPU0. Transformer block 0 stays on GPU0; blocks 1–11 and the output head run on GPU1. The transformer result returns to GPU0 for the scheduler/latent path, then the VAE input is transferred to GPU1 for decoding.

### Tiếng Việt
Text encoder chạy trên GPU0. Transformer block 0 nằm trên GPU0; các block 1–11 và output head nằm trên GPU1. Output của transformer quay về GPU0 cho scheduler/latent path, sau đó VAE input được chuyển sang GPU1 để decode.

```text
Prompt / Prompt
   |
Text Encoder / Bộ mã hóa văn bản
   cuda:0
   |
Transformer
   |
block 0 ----------------------------- cuda:0
   |
activation 0 -> 1 ------------------>
   |
blocks 1..11 ------------------------ cuda:1
norm_out / proj_out ----------------- cuda:1
   |
transformer output 1 -> 0 ---------->
   |
Latent / scheduler path ------------- cuda:0
   |
VAE input 0 -> 1 ------------------->
   |
VAE --------------------------------- cuda:1
   |
512×512 RGB image
```

Across four denoising steps, the block sequence `[0..11]` repeats four times. That repetition is expected; it is not a duplicate-block error.

Qua bốn denoising step, chuỗi block `[0..11]` lặp lại bốn lần. Đây là hành vi đúng, không phải lỗi duplicate block.


## Step 1 — Clone the pinned public source / Clone source public đã pin

### English
This cell clones the public repository at release tag `v1.0.0` into a fresh `/kaggle/working` directory, verifies the origin URL and exact release tag, records the resolved Git commit SHA, then establishes `PROJECT_ROOT` and `sys.path`.

**Expected:** `PROJECT_SOURCE_BOOTSTRAP=PASS`.  
**If it fails:** stop; do not delete a stale checkout and retry inside the same publication run.

### Tiếng Việt
Cell này clone public repository tại release tag `v1.0.0` vào một thư mục mới trong `/kaggle/working`, xác minh origin URL và đúng release tag, ghi lại Git commit SHA thực tế, sau đó mới thiết lập `PROJECT_ROOT` và `sys.path`.

**Kỳ vọng:** `PROJECT_SOURCE_BOOTSTRAP=PASS`.  
**Nếu lỗi:** dừng lại; không xóa checkout cũ rồi chạy lại trong cùng publication run.


In [ ]:
# --- Public Git source bootstrap ---
# No publication ZIP or identity JSON is required.

import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/dangkhoa2016/Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU.git"
RELEASE_REF = "v1.0.0"
PROJECT_ROOT = Path("/kaggle/working/Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU").resolve()

if PROJECT_ROOT.exists():
    raise RuntimeError(
        "PROJECT_SOURCE_TARGET_STALE: expected a fresh checkout target; "
        f"already exists: {PROJECT_ROOT}"
    )

subprocess.run(
    [
        "git", "clone",
        "--branch", RELEASE_REF,
        "--single-branch",
        REPOSITORY_URL,
        str(PROJECT_ROOT),
    ],
    check=True,
)

origin = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "remote", "get-url", "origin"],
    text=True,
).strip()
resolved_tag = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "describe", "--tags", "--exact-match"],
    text=True,
).strip()
resolved_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

if origin != REPOSITORY_URL:
    raise RuntimeError(f"PROJECT_SOURCE_ORIGIN_MISMATCH: {origin}")
if resolved_tag != RELEASE_REF:
    raise RuntimeError(
        f"PROJECT_SOURCE_RELEASE_REF_MISMATCH: expected={RELEASE_REF} actual={resolved_tag}"
    )
if not (PROJECT_ROOT / "mage_t4x2").is_dir():
    raise RuntimeError("PROJECT_SOURCE_LAYOUT_INVALID: mage_t4x2 missing")
if not (PROJECT_ROOT / "authority" / "r2g-runtime-baseline.json").is_file():
    raise RuntimeError("PROJECT_SOURCE_LAYOUT_INVALID: R2G baseline missing")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("REPOSITORY_URL=", origin)
print("RELEASE_REF=", resolved_tag)
print("RESOLVED_COMMIT=", resolved_commit)
print("PROJECT_ROOT=", PROJECT_ROOT)
print("PROJECT_SOURCE_BOOTSTRAP=PASS")


## Step 2 — Environment and GPU inventory / Môi trường và kiểm kê GPU

### English
This cell checks Python/PyTorch/CUDA and requires **exactly two Tesla T4 GPUs**. It does not load the model.

**Expected:** `GPU count = 2`, and both devices contain `T4`.

### Tiếng Việt
Cell này kiểm tra Python/PyTorch/CUDA và yêu cầu **đúng hai GPU Tesla T4**. Cell chưa load model.

**Kỳ vọng:** `GPU count = 2` và cả hai GPU đều là `T4`.


In [ ]:
# --- Environment and GPU inventory ---
# Do not load or preload the model in this cell.
# PROJECT_ROOT / sys.path were already established by the source-bootstrap cell.

import platform
import sys

assert PROJECT_ROOT is not None and (PROJECT_ROOT / "mage_t4x2").is_dir(), (
    "PROJECT_ROOT must be established by the source-bootstrap cell"
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from mage_t4x2.environment import cuda_inventory

inventory = cuda_inventory()
print(f"Python      : {platform.python_version()}")
print(f"PyTorch     : {torch.__version__} (CUDA {torch.version.cuda}) available={torch.cuda.is_available()}")
print(f"GPU count   : {len(inventory)}")
for dev in inventory:
    print(f"  GPU{dev['index']}  {dev['name']}  compute {dev['compute_capability']}  vram {dev['total_vram_bytes'] / 2**30:.1f} GiB")
assert len(inventory) == 2 and all("T4" in dev["name"] for dev in inventory), "exactly two Tesla T4 GPUs required; fail closed"

## Step 3 — R2G runtime integrity / Tính toàn vẹn runtime R2G

### English
Before public inference, the notebook verifies the frozen R2G runtime baseline. This protects the already-qualified runtime from silent modification.

**Expected:** `R2G_RUNTIME_BASELINE_INTEGRITY=PASS`, `ROWS=31`, `MISMATCHES=0`.

### Tiếng Việt
Trước khi inference public, notebook xác minh R2G runtime baseline đã đóng băng. Bước này bảo vệ runtime đã được kiểm định khỏi thay đổi ngoài ý muốn.

**Kỳ vọng:** `R2G_RUNTIME_BASELINE_INTEGRITY=PASS`, `ROWS=31`, `MISMATCHES=0`.


In [ ]:
# --- Source / runtime integrity ---
# Read-only verification of the R2G runtime baseline (authority/r2g-runtime-baseline.json).

from mage_t4x2.r2g_runtime_baseline import verify_r2g_runtime_baseline

baseline = verify_r2g_runtime_baseline(PROJECT_ROOT)
print(f"R2G_RUNTIME_BASELINE_INTEGRITY={baseline['status']}")
print(f"ROWS={len(baseline['entries'])}")
print(f"MISMATCHES={sum(1 for e in baseline['entries'] if e['status'] != 'MATCH')}")
print(f"manifest_sha256={baseline.get('manifest_sha256')}")
assert baseline["status"] == "PASS", "runtime baseline integrity failed; stop and preserve evidence"
